# legal-expand 1.6.0 - Demo interactiva para Google Colab

[![PyPI](https://img.shields.io/pypi/v/legal-expand?label=PyPI)](https://pypi.org/project/legal-expand/)
[![GitHub Release](https://img.shields.io/github/v/release/686f6c61/pypi-legal-expand?label=GitHub%20Release)](https://github.com/686f6c61/pypi-legal-expand/releases)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/686f6c61/pypi-legal-expand/blob/main/legal_expand_demo.ipynb)

Este notebook muestra la version Python de `legal-expand`: expansion de siglas juridicas espanolas, diagnostico, glosarios, auditoria, CLI, procesamiento de documentos, diccionarios personalizados y resolucion de referencias del BOE y de EUR-Lex.

**Novedades de 1.6.0:**

- **Resolucion de cualquier norma española del BOE** por su numero, mediante un indice local del catalogo consolidado (10.555 normas). Ya no depende solo de los aliases curados.
- **Traer el texto integro de los articulos** citados, tanto del BOE como de EUR-Lex (`mode='online'`, `include_unit_text=True`).
- **Normativa de la UE en EUR-Lex**: RGPD, reglamentos, directivas y decisiones se enlazan por su numero CELEX (estado `resolved-eurlex`) y traen su articulado. Ya no se marcan como no soportadas.
- **RGPD** se expande como sigla («Reglamento General de Proteccion de Datos»).
- Transporte HTTP inyectable en `BOEClient` para entornos sin `urllib`; correccion de seguridad (XSS) en el formatter HTML.

**Licencia:** desde 1.6.0, PolyForm Noncommercial License 1.0.0 (uso no comercial). Las versiones 1.5.x y previas siguen bajo MIT.

## 1. Instalacion

En Colab conviene instalar la version exacta publicada para que la demo sea reproducible.

In [ ]:
# Instala la version publicada en PyPI. Es reproducible e incluye el
# enriquecimiento online del BOE y EUR-Lex (texto integro de los articulos).
!pip install -U "legal-expand==1.6.0" -q

In [ ]:
import json
from pathlib import Path
from IPython.display import HTML, display

import legal_expand
from legal_expand import (
    boe_overrides_template,
    boe_report_by_paragraph_markdown,
    boe_report_to_html,
    boe_report_to_markdown,
    ExpansionOptions,
    auditar_texto,
    benchmark_texto,
    buscar_sigla,
    configurar_globalmente,
    detectar_referencias_boe,
    expandir_siglas,
    expandir_siglas_detallado,
    extraer_siglas,
    exportar_glosario,
    generar_glosario,
    listar_siglas,
    obtener_configuracion_global,
    obtener_estadisticas,
    obtener_info_diccionario,
    procesar_directorio,
    resetear_configuracion,
    revisar_boe,
)

print('legal-expand version:', legal_expand.__version__)
assert legal_expand.__version__ == '1.6.0'

## 2. Expansion basica

`expandir_siglas()` devuelve texto plano por defecto, anadiendo el significado entre parentesis.

In [ ]:
texto = 'La AEAT revisa el IVA segun el BOE y el art. 123 del CC.'
print(expandir_siglas(texto))


## 3. Variantes dinamicas

La libreria detecta variantes frecuentes sin inflar el diccionario fuente: mayusculas, minusculas y formas con puntos.

In [ ]:
ejemplos = [
    'La AEAT notifica.',
    'La aeat notifica.',
    'La A.E.A.T. notifica.',
    'El art 5 del CC.',
    'El art. 5 del CC.',
]

for ejemplo in ejemplos:
    print('-', expandir_siglas(ejemplo))


## 4. Formatos de salida

`plain` es el formato por defecto. Tambien puedes pedir HTML semantico o salida estructurada.

In [ ]:
muestra = 'La AEAT publica criterios sobre IVA en el BOE.'

print('PLAIN')
print(expandir_siglas(muestra))

print('\nHTML')
html = expandir_siglas(muestra, ExpansionOptions(format='html'))
display(HTML(f'<div style="font-size:16px; line-height:1.6">{html}</div>'))


In [ ]:
estructurado = expandir_siglas(muestra, ExpansionOptions(format='structured'))

print('Texto expandido:', estructurado.expanded_text)
print('Siglas detectadas:', estructurado.stats.total_acronyms_found)
for item in estructurado.acronyms:
    print(f'- {item.acronym}: {item.expansion} [{item.position.start}-{item.position.end}]')


## 5. Diagnostico de omisiones

`expandir_siglas_detallado()` explica por que una sigla se omitio: filtros, repeticion, contexto protegido, ambiguedad o no encontrada.

In [ ]:
texto_diagnostico = (
    'AEAT y BOE aparecen aqui.\n'
    'Contacto: info@aeat.es\n'
    'Codigo: `AEAT`\n'
    'XYZ no esta en el diccionario.\n'
    'AEAT aparece otra vez.'
)

diagnostico = expandir_siglas_detallado(
    texto_diagnostico,
    ExpansionOptions(expand_only_first=True, exclude=['BOE'])
)

print(diagnostico.expanded_text)
print('\nOmisiones:')
for item in diagnostico.omitted_acronyms:
    print(f'- {item.acronym} | {item.reason} | pos={item.position.start}')


## 6. Extraccion sin modificar el texto

`extraer_siglas()` detecta siglas conocidas y candidatos desconocidos. Es util para auditoria documental.

In [ ]:
extraccion = extraer_siglas('AEAT, BOE, XYZ y AEAT de nuevo.')

for item in extraccion.acronyms:
    estado = 'conocida' if item.known else 'desconocida'
    print(
        f'{item.acronym:6} {estado:12} '
        f'ocurrencia {item.occurrence_index}/{item.total_occurrences} '
        f'-> {item.expansion}'
    )


## 7. Glosarios

Desde `1.3.0` ya no hace falta construir glosarios a mano: usa `generar_glosario()` o `exportar_glosario()`.

In [ ]:
texto_glosario = 'La AEAT gestiona IVA e IRPF. El BOE publica el CC.'

print('Entradas de glosario:')
for entry in generar_glosario(texto_glosario):
    print(f'- {entry.acronym}: {entry.expansion} ({entry.count})')

print('\nMarkdown:')
print(exportar_glosario(texto_glosario, 'markdown'))

print('\nCSV:')
print(exportar_glosario(texto_glosario, 'csv'))


## 8. Auditoria completa

`auditar_texto()` combina extraccion, glosario, conocidas/desconocidas, omitidas y repetidas.

In [ ]:
reporte = auditar_texto('AEAT, BOE, XYZ, AEAT y el correo info@boe.es')

print(reporte.to_json(indent=2)[:1200])
print('\nResumen:')
print(reporte.stats)


## 9. CLI en Colab

Al instalar el paquete queda disponible el comando `legal-expand`.

In [ ]:
!legal-expand info


In [ ]:
!printf 'AEAT y BOE\n' | legal-expand --format plain


In [ ]:
%%bash
cat > sentencia.txt <<'TXT'
La AEAT liquida el IVA. El BOE publica la norma. XYZ queda pendiente.
TXT

legal-expand audit sentencia.txt --report-format markdown --output audit.md
legal-expand glossary sentencia.txt --glossary-format csv --output glosario.csv

echo '--- audit.md ---'
cat audit.md
echo '--- glosario.csv ---'
cat glosario.csv


## 10. Enriquecimiento BOE y EUR-Lex

Esta funcion no interpreta juridicamente el documento. Detecta referencias explicitas, enlaza normas cuando hay suficiente seguridad y marca como `ambiguous`, `not-found` o `review-required` lo que no debe resolver automaticamente.

El modo por defecto es `offline`: no consulta red. Resuelve las normas espanolas con aliases curados y con el indice del catalogo consolidado (cualquier norma por su numero), y la normativa de la UE a su pagina en EUR-Lex por su numero CELEX (estado `resolved-eurlex`). Para traer el texto integro de los articulos, tanto del BOE como de EUR-Lex, se usa `BOEOptions(mode='online', include_unit_text=True)`.

El flujo de revision permite ver cada referencia con explicacion, accion sugerida, HTML, informe por parrafos o plantilla editable de overrides.

In [ ]:
texto_boe = '''
La parte actora invoca el art. 217 LEC y el articulo 24 de la Constitucion Espanola.
La Ley 2/2023 se cita sin fecha, asi que debe quedar ambigua.
El art. 14.2.a) de la Ley 39/2015 se coordina con el art. 6 RGPD.
'''

informe_boe = detectar_referencias_boe(texto_boe)
revision_boe = revisar_boe(informe_boe)

print('--- JSON completo ---')
print(informe_boe.to_json(indent=2))

print('--- Revision explicada ---')
print(revision_boe.to_json(indent=2))

print('--- Markdown ---')
print(boe_report_to_markdown(informe_boe))

print('--- Informe por parrafos ---')
print(boe_report_by_paragraph_markdown(informe_boe))

print('--- Plantilla de overrides ---')
print(json.dumps(boe_overrides_template(informe_boe), ensure_ascii=False, indent=2))

display(HTML(boe_report_to_html(informe_boe)))

In [ ]:
casos_boe = [
    ('sentencia_lec', 'La parte actora invoca el art. 217 LEC.'),
    ('constitucion', 'Se vulnera el articulo 24 de la Constitucion Espanola.'),
    ('codigo_civil', 'Rige el art. 9 del Codigo Civil.'),
    ('lecrim', 'Conforme al art. 118 LECrim, procede informar derechos.'),
    ('ley_39_norma', 'La Ley 39/2015 regula el procedimiento administrativo.'),
    ('real_decreto_completo', 'Se aplico el Real Decreto 463/2020 durante el estado de alarma.'),
    ('ley_2_2023_ambigua', 'Vease el articulo 42 de la Ley 2/2023.'),
    ('ley_2_2023_con_fecha', 'Vease el articulo 42 de la Ley 2/2023, de 20 de febrero.'),
    ('multiples_normas_ambiguas', 'Los arts. 14 y 15 de las Leyes 39/2015 y 40/2015 son relevantes.'),
    ('rango_articulos_lpaca', 'Los arts. 13 a 15 LPACAP son materia de examen.'),
    ('articulo_y_siguientes', 'El art. 14 y ss. de la Ley 39/2015 debe repasarse.'),
    ('articulo_con_subletra', 'El art. 14.2.a) de la Ley 39/2015 se cita literalmente.'),
    ('articulo_bis', 'El articulo 14 bis de la Ley 39/2015 deberia detectarse.'),
    ('rgpd_sigla_no_boe', 'La base juridica es el art. 6 RGPD.'),
    ('reglamento_ue_no_boe', 'La base juridica es el articulo 6 del Reglamento (UE) 2016/679.'),
    ('orden_anexo', 'Debe cumplirse el anexo I de la Orden HFP/1030/2021.'),
    ('disposicion_final_larga', 'Se aplica la disposicion final septima de la Ley 2/2023, de 20 de febrero.'),
    ('disposicion_final_abreviada', 'Se aplica la disp. final septima de la Ley 2/2023, de 20 de febrero.'),
    ('real_decreto_abreviado', 'El art. 3 del RD 203/2021 regula la actuacion administrativa automatizada.'),
    ('ley_organica_abreviada_mas_rgpd', 'El art. 6.1 de la LO 3/2018 debe coordinarse con el RGPD.'),
    ('inferencia_una_norma_mismo_parrafo', 'La Ley 39/2015 regula la relacion electronica. En su art. 14 se establece la obligacion.'),
    ('no_infiere_entre_parrafos', 'La Ley 39/2015 regula la relacion electronica.\n\nEl art. 14 se cita despues.'),
    ('protege_url_y_codigo', 'No tocar https://www.boe.es/buscar/act.php?id=BOE-A-2015-10565 ni `art. 14 Ley 39/2015`.'),
    ('no_captura_stc_ni_procedimiento', 'La STC 39/2015 no debe capturarse. Tampoco el procedimiento 39/2015.'),
    ('boe_id_directo', 'Referencia oficial BOE-A-2015-10565.'),
]

for nombre, texto in casos_boe:
    salida = detectar_referencias_boe(texto)
    revision = revisar_boe(salida)
    resumen = [
        {
            'status': ref.status,
            'texto': ref.original_text,
            'unidad': ref.unit_text,
            'boe_id': ref.norm.boe_id if ref.norm else None,
            'motivo': ref.reason,
            'explicacion': revision.items[i].explanation,
        }
        for i, ref in enumerate(salida.references)
    ]
    print(f'\n## {nombre}')
    print(json.dumps(resumen, ensure_ascii=False, indent=2))


## 10 ter. Resolver cualquier norma por su numero (indice del catalogo)

Desde 1.6.0, ademas de los aliases curados, el paquete resuelve cualquier norma española del BOE por su rango y numero oficial usando un indice local del catalogo consolidado. Funciona **offline** y cubre miles de normas que antes quedaban como `needs-boe-search`.

In [ ]:
# Normas que NO estan en los aliases curados: se resuelven por el indice del catalogo.
casos_indice = [
    'La Ley 19/2013 de transparencia obliga a publicar informacion.',
    'El Real Decreto 1112/2018 sobre accesibilidad de sitios web del sector publico.',
    'La Ley 7/1985 reguladora de las Bases del Regimen Local.',
    'El Real Decreto-ley 8/2020 de medidas urgentes extraordinarias.',
    'La Ley Organica 10/1995 del Codigo Penal.',
]
for texto in casos_indice:
    for ref in detectar_referencias_boe(texto).references:
        boe_id = ref.norm.boe_id if ref.norm else '-'
        titulo = ref.norm.title if ref.norm else ''
        print(f'{ref.status:18} {ref.original_text:32} -> {boe_id} | {titulo}')

## 10 quater. Traer el texto integro del articulo (BOE y EUR-Lex, online)

Con `--mode online` en la CLI (o `BOEOptions(mode='online', include_unit_text=True)` en Python) `legal-expand` descarga el **texto integro** del articulo citado: del BOE para la norma espanola y de EUR-Lex para la normativa de la UE. En Colab hay red directa, no hace falta proxy. El informe en Markdown inserta cada articulo bajo su encabezado (`### Articulo N`).

In [ ]:
%%bash
# --mode online descarga el texto integro de cada articulo citado:
# del BOE (Ley 39/2015) y de EUR-Lex (RGPD). Colab tiene red directa.
cat > boe_demo.txt <<'TXT'
La notificacion se rige por el art. 14.2.a) de la Ley 39/2015.
El tratamiento de datos personales se ampara en el art. 6 del RGPD.
TXT

legal-expand boe boe_demo.txt --mode online --report-format markdown

In [ ]:
overrides_boe = {
    'references': [
        {
            'text': 'la norma especial de informantes',
            'boe_id': 'BOE-A-2023-4513',
            'title': 'Ley 2/2023, de 20 de febrero',
            'unit': 'articulo 42',
        }
    ]
}

texto_manual = 'Debe revisarse la norma especial de informantes antes de cerrar el informe.'
manual = detectar_referencias_boe(texto_manual, overrides=overrides_boe)
print(boe_report_to_markdown(manual))
print(revisar_boe(manual).to_json(indent=2))

## 10 quinquies. Calidad de la release 1.6.0

`1.6.0` amplia el enriquecimiento BOE: resuelve cualquier norma española por su numero (indice del catalogo consolidado) y trae el texto integro de los articulos citados. La suite de tests pasa al **100%** de cobertura, con `mypy`, `ruff` y `bandit` limpios.

In [ ]:
calidad_160 = {
    'coverage': '100% (statements y branches)',
    'tests': 'suite completa (mypy, ruff, bandit limpios)',
    'package': 'legal-expand==1.6.0',
    'boe_index': '10.555 normas del catalogo consolidado',
}
print(json.dumps(calidad_160, ensure_ascii=False, indent=2))

## 11. Documentos y batch

`procesar_directorio()` procesa carpetas de `.txt`, `.md` y `.html`. En HTML expande nodos de texto conservando etiquetas y atributos.

In [ ]:
entrada = Path('demo_docs')
salida = Path('demo_docs_expandidos')
entrada.mkdir(exist_ok=True)

(entrada / 'nota.txt').write_text('La AEAT revisa el IVA publicado en el BOE.\n', encoding='utf-8')
(entrada / 'web.html').write_text(
    '<p>La AEAT revisa el IVA.</p><a href="https://aeat.es">AEAT</a>',
    encoding='utf-8'
)

resultados = procesar_directorio(entrada, salida, ExpansionOptions(), document_format='auto')
for result in resultados:
    print(result.input_path, '->', result.output_path, 'OK=', result.processed)

print('\nTXT expandido:')
print((salida / 'nota.txt').read_text(encoding='utf-8'))

print('HTML expandido:')
print((salida / 'web.html').read_text(encoding='utf-8'))


## 12. Diccionarios personalizados JSON/CSV

Puedes cargar siglas propias sin tocar el diccionario base. No se usan perfiles ni categorias: solo entradas personalizadas directas.

In [ ]:
custom_json = Path('custom_legal_expand.json')
custom_json.write_text(json.dumps([
    {
        'original': 'LXP',
        'significado': 'Legal Expand Python',
        'variants': ['L.X.P.'],
        'source': 'demo-colab',
        'keywords': ['paquete', 'python', 'demo'],
        'priority': 200,
    }
], ensure_ascii=False, indent=2), encoding='utf-8')

opciones_custom = ExpansionOptions(custom_dictionaries=[str(custom_json)])
print(expandir_siglas('LXP y L.X.P. aparecen junto a AEAT.', opciones_custom))

info = obtener_info_diccionario([str(custom_json)])
print(info.to_json(indent=2))


## 13. Busqueda y estadisticas del diccionario

In [ ]:
print('Buscar AEAT:')
print(buscar_sigla('AEAT').to_json(indent=2))

siglas = listar_siglas()
print('Total listadas:', len(siglas))
print('Primeras 20:', siglas[:20])

print('\nEstadisticas:')
print(obtener_estadisticas().to_json(indent=2))


## 14. Configuracion global

La configuracion global permite definir defaults para una aplicacion completa. Reseteala al final de la demo para evitar sorpresas en celdas posteriores.

In [ ]:
from legal_expand import GlobalConfig

configurar_globalmente(GlobalConfig(
    default_options=ExpansionOptions(format='html', expand_only_first=True)
))

print(obtener_configuracion_global().to_json(indent=2))
display(HTML(expandir_siglas('AEAT, AEAT y BOE')))

resetear_configuracion()
print('Configuracion reseteada:', obtener_configuracion_global().to_json())


## 15. Benchmark rapido

In [ ]:
bench = benchmark_texto(
    'La AEAT gestiona el IVA segun el BOE y el CC. ' * 50,
    iterations=100,
)
print(bench.to_json(indent=2))


## 16. Instalacion desde PyPI y enlaces

- PyPI: https://pypi.org/project/legal-expand/
- GitHub: https://github.com/686f6c61/pypi-legal-expand
- npm original: https://www.npmjs.com/package/legal-expand

Para instalar en un proyecto normal:

```bash
pip install legal-expand
```

Para usar la CLI:

```bash
legal-expand info
legal-expand audit documento.txt --report-format markdown
legal-expand batch docs/ docs-expandidos/ --format html
legal-expand boe documento.txt --report-format markdown
legal-expand boe documento.txt --report-format review-json
legal-expand boe documento.txt --overrides-template
```